# N-Body Orbital Mechanics Simulator — Phase 1: Earth–Sun Orbit

**Author:** Andro J. Romero C.  
**Status:** Work in progress — v0.1 (fixed-Sun, two-body approximation)

## Objective
Build a numerical simulation of Earth's orbit using Newtonian gravitation and **forward Euler integration**. This is a stepping stone toward a validated, uncertainty-aware N-body simulator.

**Current model assumptions:** The Sun is fixed at the origin; Earth is treated as a test body; motion is two-dimensional; only Newtonian solar gravity is included. This notebook is **not yet** an interacting N-body model.

**Units:** SI (meters, kilograms, seconds).

## 1 — Import packages

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 2 — Physical constants and initial conditions

In [ ]:
# Gravitational constant (m^3 kg^-1 s^-2)
G = 6.67430e-11

# Mass of the Sun (kg)
M_sun = 1.989e30

# Initial Earth–Sun distance (m)
x_0 = 1.496e11

# Circular orbital speed for this idealized model (m/s)
v_orbital = np.sqrt(G * M_sun / x_0)

print(f"Circular orbital speed: {v_orbital:.4f} m/s")

## 3 — Derive gravitational acceleration

Newton's gravitational force has magnitude

$$F = \frac{GM_\odot m}{r^2}.$$

Using $F=ma$ and cancelling Earth's mass gives

$$a=\frac{GM_\odot}{r^2}.$$

The **vector** acceleration points toward the Sun:

$$\boxed{\mathbf a(\mathbf r)=-\frac{GM_\odot}{|\mathbf r|^3}\mathbf r}.$$

In Cartesian coordinates:

$$a_x=-\frac{GM_\odot x}{(x^2+y^2)^{3/2}},\qquad a_y=-\frac{GM_\odot y}{(x^2+y^2)^{3/2}}.$$

## 4 — Evaluate the initial gravitational acceleration

In [ ]:
a_magnitude = G * M_sun / x_0**2
print(f"Initial acceleration magnitude: {a_magnitude:.8f} m/s²")

## 5 — Circular orbital velocity

For a circular orbit, gravity supplies the centripetal acceleration:

$$\frac{GM_\odot}{r^2}=\frac{v^2}{r} \quad\Rightarrow\quad \boxed{v=\sqrt{\frac{GM_\odot}{r}}}.$$

We computed this value above and will use it consistently in the initial velocity vector.

## 6 — Represent position and velocity as vectors

In [ ]:
# Earth starts on the positive x-axis and moves in the positive y-direction.
r_0 = np.array([x_0, 0.0], dtype=float)
v_0 = np.array([0.0, v_orbital], dtype=float)

print("Initial position (m):", r_0)
print("Initial velocity (m/s):", v_0)

## 7 — Write the gravitational acceleration function

In [ ]:
def grav_acc(r):
    """Acceleration from a fixed Sun at the origin (m/s²)."""
    r_mag = np.linalg.norm(r)
    if r_mag == 0:
        raise ValueError("Position cannot be at the Sun's center (r = 0).")
    return -(G * M_sun / r_mag**3) * r


a_0 = grav_acc(r_0)
print("Initial acceleration vector (m/s²):", a_0)
print("Initial acceleration magnitude (m/s²):", np.linalg.norm(a_0))

## 8 — Advance one time step by forward Euler

For a time step $\Delta t$:

$$\boxed{\mathbf r_{n+1}=\mathbf r_n+\mathbf v_n\Delta t}$$

$$\boxed{\mathbf v_{n+1}=\mathbf v_n+\mathbf a(\mathbf r_n)\Delta t}$$

Both updates use the **old** state at step $n$.

In [ ]:
dt = 3600.0  # one hour, in seconds

r_1 = r_0 + v_0 * dt
v_1 = v_0 + a_0 * dt
a_1 = grav_acc(r_1)

print("Position after one hour (m):", r_1)
print("Velocity after one hour (m/s):", v_1)
print("Acceleration after one hour (m/s²):", a_1)

## 9 — Automate the updates for one year

In [ ]:
# Restart from the same initial conditions every time this cell runs.
r = r_0.copy()
v = v_0.copy()

dt = 3600.0               # seconds per step
n_steps = 365 * 24        # 8,760 one-hour steps = 365 days

# Store the initial state as well as each updated state.
positions = [r.copy()]

for step in range(n_steps):
    a = grav_acc(r)

    # Forward Euler updates, both based on the current state.
    r_new = r + v * dt
    v_new = v + a * dt

    r = r_new
    v = v_new
    positions.append(r.copy())

positions = np.array(positions)

print("Position array shape:", positions.shape)
print("First position (m):", positions[0])
print("Last position (m):", positions[-1])
assert positions.shape == (8761, 2)

## 10 — Extract the x and y coordinates

In [ ]:
x = positions[:, 0]  # all x-coordinates
y = positions[:, 1]  # all y-coordinates

## 11 — Plot the simulated trajectory

In [ ]:
plt.figure(figsize=(7, 7))
plt.plot(x, y, label="Earth trajectory (forward Euler)")
plt.scatter(0, 0, color="orange", s=110, label="Sun (fixed)", zorder=3)
plt.scatter(x[0], y[0], color="green", s=45, label="Initial Earth position", zorder=3)
plt.xlabel("x position (m)")
plt.ylabel("y position (m)")
plt.title("Earth–Sun Orbit — 365 Days (Forward Euler)")
plt.axis("equal")
plt.grid(True)
plt.legend()
plt.show()

## 12 — Interpretation and next steps

The orbit should be **approximately circular**, but the forward Euler method introduces numerical error. In particular, it generally does **not** conserve orbital energy and can cause secular orbital drift. A non-closed trajectory is not necessarily a programming mistake.

**Next milestones:**
1. Record velocity at every step and evaluate energy and angular-momentum drift.
2. Replace forward Euler with velocity Verlet (a symplectic integrator).
3. Extend the force calculation to several mutually interacting bodies.
4. Test time-step convergence and compare against Keplerian predictions.
5. Add Monte Carlo propagation of initial-condition uncertainty.

**Reproducibility check:** In Jupyter, choose **Kernel → Restart Kernel and Run All Cells**; this notebook should run from top to bottom without relying on variables from previous sessions.